# Disagregasi Tanaman Pangan Kabupaten Kediri (versi perbaikan)

Notebook ini membentuk dataset **kecamatan × komoditas** (26 kecamatan) dari total Kabupaten.

**Sumber input** (folder `data/`):
- `Tanaman Pangan.xlsx` : luas panen, produksi, produktivitas tingkat **Kabupaten** (BPS)
- `Luas Lahan Per Kecamatan.csv` : luas sawah dan lahan pertanian bukan sawah per kecamatan (BPS)

**Metode:** luas panen dan produksi Kabupaten dialokasikan ke kecamatan secara **proporsional**.
Pembobot per komoditas bisa diatur (`BOBOT_SAWAH`): padi memakai luas sawah, umbi-umbian memakai lahan
pertanian bukan sawah, dan komoditas lain (kedelai, kacang-kacangan, jagung) memakai campuran keduanya.

**Output** (folder `data/derived/`):
1. `Tanaman Pangan Per Kecamatan.csv` : hasil **estimasi** per kecamatan (ada `tahun`, `is_estimate`, `metode_alokasi`, `sumber`)
2. `Tanaman Pangan Kabupaten.csv` : data **asli** BPS tingkat Kabupaten (tabel terpisah)
3. `Master Kecamatan.csv` : kode dan nama baku kecamatan, sebagai kunci join

> **Catatan:** hasil tingkat kecamatan adalah **estimasi, bukan data observasi**. Jangan dipakai untuk
> klaim akurasi model. Ganti dengan data asli DKPP/BPS per kecamatan begitu tersedia.

In [ ]:
# 1. IMPORT LIBRARY
import os
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

print("✓ Library berhasil di-import.")

In [ ]:
# 2. KONFIGURASI  (ubah bagian ini saja)

# WAJIB DIISI: tahun data BPS yang dipakai, misalnya 2023.
# Tanpa tahun, data tidak bisa dipakai untuk deret waktu (prediksi).
TAHUN_DATA = None

SUMBER_TANAMAN = "BPS Kabupaten Kediri - Tanaman Pangan"
SUMBER_LAHAN = "BPS Kabupaten Kediri - Penggunaan Lahan"

NAMA_FILE_TANAMAN = "Tanaman Pangan.xlsx"
NAMA_FILE_LAHAN = "Luas Lahan Per Kecamatan.csv"
SKIPROWS_LAHAN = 4            # jumlah baris judul yang dilewati di CSV lahan
JUMLAH_KECAMATAN = 26

# Penyeragaman nama komoditas (kunci = huruf kecil).
NAMA_KOMODITAS_BAKU = {
    "padi": "Padi",
    "jagung": "Jagung",
    "kedelai": "Kedelai",
    "kacang kedelai": "Kedelai",
    "kacang tanah": "Kacang Tanah",
    "kacang hijau": "Kacang Hijau",
    "ubi jalar": "Ubi Jalar",
    "ketela rambat": "Ubi Jalar",
    "ubi kayu": "Ubi Kayu",
    "ketela pohon": "Ubi Kayu",
    "singkong": "Ubi Kayu",
}

# Komoditas yang disebut di proposal (dipakai untuk peringatan kelengkapan).
KOMODITAS_PROPOSAL = ["Padi", "Jagung", "Kedelai", "Kacang Tanah",
                      "Kacang Hijau", "Ubi Kayu", "Ubi Jalar"]

# Pembobot alokasi: 1.0 = sepenuhnya luas sawah, 0.0 = sepenuhnya lahan pertanian bukan sawah.
# INI ASUMSI. Kedelai/kacang-kacangan sering ditanam di sawah setelah padi, sedangkan umbi-umbian
# umumnya di tegalan. Validasi dengan DKPP dan sesuaikan angkanya.
BOBOT_SAWAH = {
    "Padi": 1.0,
    "Ubi Kayu": 0.0,
    "Ubi Jalar": 0.0,
}
BOBOT_SAWAH_DEFAULT = 0.5     # untuk komoditas yang tidak tercantum di atas

if TAHUN_DATA is None:
    print("⚠️  TAHUN_DATA belum diisi. Kolom `tahun` akan kosong. Isi dulu sebelum dipakai untuk analisis.")
else:
    print(f"✓ Tahun data: {TAHUN_DATA}")

In [ ]:
# 3. MENENTUKAN LOKASI PROJECT DAN DATASET

CURRENT_DIR = os.getcwd()
PROJECT_ROOT = None

# Cari folder "data" dari current directory dan parent-nya.
search_dir = CURRENT_DIR
while True:
    if os.path.isdir(os.path.join(search_dir, "data")):
        PROJECT_ROOT = search_dir
        break
    parent_dir = os.path.dirname(search_dir)
    if parent_dir == search_dir:
        break
    search_dir = parent_dir

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Folder 'data' tidak ditemukan. "
        "Pastikan notebook dijalankan dari dalam repository AgriDash."
    )

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
OUTPUT_DIR = os.path.join(DATA_DIR, "derived")
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILE_TANAMAN = os.path.join(DATA_DIR, NAMA_FILE_TANAMAN)
FILE_LAHAN = os.path.join(DATA_DIR, NAMA_FILE_LAHAN)

for f in (FILE_TANAMAN, FILE_LAHAN):
    if not os.path.isfile(f):
        raise FileNotFoundError(f"File tidak ditemukan: {f}")

print("Project root   :", PROJECT_ROOT)
print("Data directory :", DATA_DIR)
print("Output         :", OUTPUT_DIR)
print("✓ Semua file input ditemukan.")

In [ ]:
# 4. FUNGSI BANTU

def extract_number(value):
    """Ubah teks/angka apa pun menjadi float. Mengenali format Indonesia (1.234,56) dan US (1,234.56).

    Catatan: teks dengan satu titik saja (mis. '1.234') dianggap desimal, bukan pemisah ribuan.
    Teks dengan koma saja dianggap pemisah ribuan jika berpola 1,391 / 12,345; selain itu koma desimal.
    """
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    s = re.sub(r"[^0-9.,-]", "", str(value).strip())
    if s in {"", "-", ".", ","}:
        return np.nan

    if "," in s and "." in s:
        # Pemisah desimal = tanda yang muncul paling akhir.
        if s.rfind(",") > s.rfind("."):
            s = s.replace(".", "").replace(",", ".")
        else:
            s = s.replace(",", "")
    elif "," in s:
        if re.fullmatch(r"-?[1-9]\d{0,2}(,\d{3})+", s):
            s = s.replace(",", "")      # 1,391 atau 12,345,678 = pemisah ribuan
        else:
            s = s.replace(",", ".")     # 0,75 atau 12,5 = koma desimal

    try:
        return float(s)
    except ValueError:
        return np.nan


def baku_komoditas(nama):
    kunci = str(nama).strip().lower()
    return NAMA_KOMODITAS_BAKU.get(kunci, str(nama).strip().title())


print("✓ Fungsi bantu siap.")

In [ ]:
# 5. BACA DATA TANAMAN PANGAN KABUPATEN

df_tanaman_raw = pd.read_excel(FILE_TANAMAN)

print("Ukuran data:", df_tanaman_raw.shape)
display(df_tanaman_raw)

In [ ]:
# 6. CLEANING DATA TANAMAN PANGAN

required = {"Tanaman", "Luas Panen", "Produksi", "Produktivitas"}
missing = required - set(df_tanaman_raw.columns)
if missing:
    raise ValueError(f"Kolom tidak ditemukan: {missing}")

df_kab = pd.DataFrame({
    "tanaman_asli": df_tanaman_raw["Tanaman"].astype(str).str.strip(),
    "luas_panen_kab_ha": df_tanaman_raw["Luas Panen"].apply(extract_number),
    "produksi_kab_ton": df_tanaman_raw["Produksi"].apply(extract_number),
    "produktivitas_kab_kuintal_ha": df_tanaman_raw["Produktivitas"].apply(extract_number),
})
df_kab["tanaman"] = df_kab["tanaman_asli"].apply(baku_komoditas)

if df_kab[["luas_panen_kab_ha", "produksi_kab_ton"]].isna().any().any():
    raise ValueError("Ada nilai luas panen/produksi yang gagal dibaca.")

if df_kab["tanaman"].duplicated().any():
    dup = df_kab.loc[df_kab["tanaman"].duplicated(keep=False), ["tanaman_asli", "tanaman"]]
    raise ValueError(f"Nama komoditas ganda setelah dibakukan:\n{dup}")

# Cek konsistensi produktivitas bawaan BPS vs hitung ulang (ton/ha x 10 = kuintal/ha).
hitung = df_kab["produksi_kab_ton"] / df_kab["luas_panen_kab_ha"] * 10
selisih_rel = (hitung - df_kab["produktivitas_kab_kuintal_ha"]).abs() / hitung
for t, s in zip(df_kab["tanaman"], selisih_rel):
    if pd.notna(s) and s > 0.01:
        print(f"⚠️  Produktivitas {t}: selisih {s:.1%} antara nilai BPS dan hitung ulang. Cek sumbernya.")

df_kab = df_kab[["tanaman", "tanaman_asli", "luas_panen_kab_ha",
                 "produksi_kab_ton", "produktivitas_kab_kuintal_ha"]]

print("Komoditas:", df_kab["tanaman"].tolist())
display(df_kab)

In [ ]:
# 7. BACA DATA LUAS LAHAN PER KECAMATAN

# header=None penting agar baris pertama data (Mojo) tidak hilang.
df_lahan_raw = pd.read_csv(
    FILE_LAHAN,
    skiprows=SKIPROWS_LAHAN,
    header=None,
    dtype=str,
)

if df_lahan_raw.shape[1] < 5:
    raise ValueError(f"CSV lahan hanya punya {df_lahan_raw.shape[1]} kolom, butuh minimal 5.")

df_lahan_raw = df_lahan_raw.iloc[:, :5]
df_lahan_raw.columns = [
    "kecamatan_raw",
    "luas_sawah_ha",
    "lahan_pertanian_non_sawah_ha",
    "lahan_non_pertanian_ha",
    "total_lahan_ha",
]

print("Ukuran data:", df_lahan_raw.shape)
display(df_lahan_raw.head())

In [ ]:
# 8. CLEANING DATA LAHAN

df_lahan = df_lahan_raw.copy()

# Kode kecamatan BPS, mis. "[010] Mojo" -> kode "010", nama "Mojo".
df_lahan["kode_kecamatan"] = df_lahan["kecamatan_raw"].str.extract(r"^\s*\[(\d+)\]", expand=False)
df_lahan["kecamatan"] = (
    df_lahan["kecamatan_raw"].astype(str)
    .str.replace(r"^\s*\[\d+\]\s*", "", regex=True)
    .str.strip()
)

# Buang baris jumlah/total kabupaten jika ada.
baris_total = df_lahan["kecamatan"].str.contains(r"^(?:jumlah|total|kabupaten)", case=False, regex=True, na=False)
if baris_total.any():
    print("Baris total dibuang:", df_lahan.loc[baris_total, "kecamatan"].tolist())
df_lahan = df_lahan[~baris_total]

numeric_cols = ["luas_sawah_ha", "lahan_pertanian_non_sawah_ha",
                "lahan_non_pertanian_ha", "total_lahan_ha"]
for col in numeric_cols:
    df_lahan[col] = df_lahan[col].apply(extract_number)

df_lahan = df_lahan.dropna(subset=["kecamatan", "luas_sawah_ha"]).reset_index(drop=True)
df_lahan = df_lahan[["kode_kecamatan", "kecamatan"] + numeric_cols]

print("Jumlah kecamatan:", df_lahan["kecamatan"].nunique())
display(df_lahan)

In [ ]:
# 9. VALIDASI KECAMATAN

jumlah_kecamatan = df_lahan["kecamatan"].nunique()
if jumlah_kecamatan != JUMLAH_KECAMATAN:
    raise ValueError(f"Jumlah kecamatan = {jumlah_kecamatan}, bukan {JUMLAH_KECAMATAN}.")
if df_lahan["kecamatan"].duplicated().any():
    raise ValueError("Ada nama kecamatan yang duplikat.")
if "Mojo" not in df_lahan["kecamatan"].values:
    raise ValueError("Kecamatan Mojo tidak ditemukan (baris pertama mungkin hilang).")

for col in ["luas_sawah_ha", "lahan_pertanian_non_sawah_ha"]:
    if df_lahan[col].isna().any():
        raise ValueError(f"Ada nilai kosong di kolom {col}.")
    if (df_lahan[col] < 0).any():
        raise ValueError(f"Ada nilai negatif di kolom {col}.")

# Total lahan harus sama dengan jumlah tiga komponennya.
komponen = (df_lahan["luas_sawah_ha"] + df_lahan["lahan_pertanian_non_sawah_ha"]
            + df_lahan["lahan_non_pertanian_ha"])
tidak_cocok = df_lahan.loc[~np.isclose(komponen, df_lahan["total_lahan_ha"], atol=1), "kecamatan"].tolist()
if tidak_cocok:
    print("⚠️  Total lahan tidak sama dengan jumlah komponen di:", tidak_cocok)

print(f"✓ {jumlah_kecamatan} kecamatan ditemukan, tanpa duplikat.")
print(f"✓ Total luas sawah                 : {df_lahan['luas_sawah_ha'].sum():,.2f} Ha")
print(f"✓ Total lahan pertanian bukan sawah: {df_lahan['lahan_pertanian_non_sawah_ha'].sum():,.2f} Ha")

In [ ]:
# 10. HITUNG PROPORSI (SAWAH DAN LAHAN BUKAN SAWAH)

df_lahan["proporsi_sawah"] = df_lahan["luas_sawah_ha"] / df_lahan["luas_sawah_ha"].sum()
df_lahan["proporsi_non_sawah"] = (
    df_lahan["lahan_pertanian_non_sawah_ha"] / df_lahan["lahan_pertanian_non_sawah_ha"].sum()
)

assert np.isclose(df_lahan["proporsi_sawah"].sum(), 1.0)
assert np.isclose(df_lahan["proporsi_non_sawah"].sum(), 1.0)

display(df_lahan.sort_values("luas_sawah_ha", ascending=False))

In [ ]:
# 11. BENTUK KECAMATAN x KOMODITAS

df_final = df_lahan.merge(df_kab, how="cross")

jumlah_komoditas = df_kab["tanaman"].nunique()
expected_rows = jumlah_kecamatan * jumlah_komoditas
if len(df_final) != expected_rows:
    raise ValueError(f"Jumlah baris {len(df_final)} tidak sesuai target {expected_rows}.")

# Proporsi alokasi per komoditas = campuran proporsi sawah dan proporsi lahan bukan sawah.
df_final["bobot_sawah"] = df_final["tanaman"].map(BOBOT_SAWAH).fillna(BOBOT_SAWAH_DEFAULT)
df_final["proporsi_alokasi"] = (
    df_final["bobot_sawah"] * df_final["proporsi_sawah"]
    + (1 - df_final["bobot_sawah"]) * df_final["proporsi_non_sawah"]
)
df_final["metode_alokasi"] = df_final["bobot_sawah"].apply(
    lambda w: f"proporsional: {w:.0%} luas sawah + {1 - w:.0%} lahan pertanian bukan sawah"
)

print(f"✓ Dataset terbentuk: {jumlah_kecamatan} kecamatan × {jumlah_komoditas} komoditas = {len(df_final)} baris")
print("\nPembobot yang dipakai:")
display(df_final[["tanaman", "bobot_sawah"]].drop_duplicates().reset_index(drop=True))

In [ ]:
# 12. ESTIMASI LUAS PANEN, PRODUKSI, DAN PRODUKTIVITAS

df_final["luas_panen_estimasi_ha"] = df_final["luas_panen_kab_ha"] * df_final["proporsi_alokasi"]
df_final["produksi_estimasi_ton"] = df_final["produksi_kab_ton"] * df_final["proporsi_alokasi"]

# Ton/Ha menjadi Kuintal/Ha dengan dikali 10.
df_final["produktivitas_estimasi_kuintal_ha"] = np.where(
    df_final["luas_panen_estimasi_ha"] > 0,
    df_final["produksi_estimasi_ton"] / df_final["luas_panen_estimasi_ha"] * 10,
    np.nan,
)

display(df_final[["kecamatan", "tanaman", "proporsi_alokasi", "luas_panen_estimasi_ha",
                  "produksi_estimasi_ton", "produktivitas_estimasi_kuintal_ha"]].head(10))

In [ ]:
# 13. VALIDASI TOTAL KEMBALI KE DATA KABUPATEN

validasi = (
    df_final.groupby("tanaman")
    .agg(
        luas_panen_kabupaten=("luas_panen_kab_ha", "first"),
        total_luas_panen_estimasi=("luas_panen_estimasi_ha", "sum"),
        produksi_kabupaten=("produksi_kab_ton", "first"),
        total_produksi_estimasi=("produksi_estimasi_ton", "sum"),
        total_proporsi=("proporsi_alokasi", "sum"),
    )
    .reset_index()
)

validasi["luas_panen_valid"] = np.isclose(
    validasi["total_luas_panen_estimasi"], validasi["luas_panen_kabupaten"], rtol=1e-9, atol=1e-8)
validasi["produksi_valid"] = np.isclose(
    validasi["total_produksi_estimasi"], validasi["produksi_kabupaten"], rtol=1e-9, atol=1e-8)
validasi["proporsi_valid"] = np.isclose(validasi["total_proporsi"], 1.0)

display(validasi)

assert df_final["kecamatan"].nunique() == JUMLAH_KECAMATAN
assert df_final["tanaman"].nunique() == jumlah_komoditas
assert len(df_final) == expected_rows
assert not df_final.duplicated(["kecamatan", "tanaman"]).any()
assert validasi["luas_panen_valid"].all()
assert validasi["produksi_valid"].all()
assert validasi["proporsi_valid"].all()

print("========== VALIDASI BERHASIL ==========")
print(f"✓ Kecamatan : {df_final['kecamatan'].nunique()}")
print(f"✓ Komoditas : {df_final['tanaman'].nunique()}")
print(f"✓ Baris     : {len(df_final)}")
print("✓ Total luas panen dan produksi hasil alokasi sama dengan data Kabupaten.")

In [ ]:
# 14. CEK KEWAJARAN DATA (peringatan, tidak menghentikan proses)

ada_peringatan = False

# a) Komoditas di proposal yang belum ada di data
kurang = [k for k in KOMODITAS_PROPOSAL if k not in set(df_kab["tanaman"])]
if kurang:
    ada_peringatan = True
    print("⚠️  Komoditas di proposal yang belum ada di data:", ", ".join(kurang))

# b) Luas panen Kabupaten sangat kecil (kemungkinan salah satuan/tahun/baris)
kecil = df_kab.loc[df_kab["luas_panen_kab_ha"] < 10, ["tanaman", "luas_panen_kab_ha"]]
for _, r in kecil.iterrows():
    ada_peringatan = True
    print(f"⚠️  {r['tanaman']}: luas panen Kabupaten hanya {r['luas_panen_kab_ha']:,.2f} ha. Cek sumber (satuan/tahun).")

# c) Luas panen padi vs luas sawah
padi = df_kab.loc[df_kab["tanaman"] == "Padi", "luas_panen_kab_ha"]
if len(padi):
    rasio = padi.iloc[0] / df_lahan["luas_sawah_ha"].sum()
    if rasio < 1:
        ada_peringatan = True
        print(f"⚠️  Luas panen padi / luas sawah = {rasio:.2f} (< 1). Umumnya di atas 1 karena panen >1 kali/tahun. "
              "Pastikan tahun dan periode kedua sumber sama.")

# d) Estimasi kecamatan terlalu kecil untuk bermakna
sangat_kecil = (df_final["luas_panen_estimasi_ha"] < 1).groupby(df_final["tanaman"]).sum()
for t, n in sangat_kecil[sangat_kecil > 0].items():
    ada_peringatan = True
    print(f"⚠️  {t}: {n} dari {jumlah_kecamatan} kecamatan punya estimasi luas panen < 1 ha (tidak bermakna).")

# e) Tahun
if TAHUN_DATA is None:
    ada_peringatan = True
    print("⚠️  TAHUN_DATA belum diisi.")

print("\nSemua estimasi tingkat kecamatan berstatus ESTIMASI (is_estimate = True).")
if not ada_peringatan:
    print("✓ Tidak ada peringatan tambahan.")

In [ ]:
# 15. SUSUN TABEL OUTPUT

tahun_val = TAHUN_DATA if TAHUN_DATA is not None else np.nan

# (1) Estimasi per kecamatan
df_out_kec = pd.DataFrame({
    "kode_kecamatan": df_final["kode_kecamatan"],
    "kecamatan": df_final["kecamatan"],
    "tahun": tahun_val,
    "tanaman": df_final["tanaman"],
    "luas_panen_ha": df_final["luas_panen_estimasi_ha"].round(2),
    "produksi_ton": df_final["produksi_estimasi_ton"].round(2),
    "produktivitas_kuintal_ha": df_final["produktivitas_estimasi_kuintal_ha"].round(2),
    "proporsi_alokasi": df_final["proporsi_alokasi"].round(6),
    "bobot_sawah": df_final["bobot_sawah"],
    "metode_alokasi": df_final["metode_alokasi"],
    "is_estimate": True,
    "sumber": f"{SUMBER_TANAMAN}; {SUMBER_LAHAN}",
}).sort_values(["tanaman", "kecamatan"]).reset_index(drop=True)

# (2) Data asli Kabupaten
df_out_kab = pd.DataFrame({
    "tahun": tahun_val,
    "tanaman": df_kab["tanaman"],
    "tanaman_asli": df_kab["tanaman_asli"],
    "luas_panen_ha": df_kab["luas_panen_kab_ha"].round(2),
    "produksi_ton": df_kab["produksi_kab_ton"].round(2),
    "produktivitas_kuintal_ha": df_kab["produktivitas_kab_kuintal_ha"].round(2),
    "is_estimate": False,
    "sumber": SUMBER_TANAMAN,
})

# (3) Master kecamatan
df_out_master = df_lahan[["kode_kecamatan", "kecamatan", "luas_sawah_ha",
                          "lahan_pertanian_non_sawah_ha", "lahan_non_pertanian_ha",
                          "total_lahan_ha"]].copy()
df_out_master["tahun_data_lahan"] = tahun_val
df_out_master["sumber"] = SUMBER_LAHAN

print("Estimasi per kecamatan:", df_out_kec.shape)
print("Data Kabupaten        :", df_out_kab.shape)
print("Master kecamatan      :", df_out_master.shape)
display(df_out_kec.head(10))

In [ ]:
# 16. SIMPAN CSV

OUT_KEC = os.path.join(OUTPUT_DIR, "Tanaman Pangan Per Kecamatan.csv")
OUT_KAB = os.path.join(OUTPUT_DIR, "Tanaman Pangan Kabupaten.csv")
OUT_MASTER = os.path.join(OUTPUT_DIR, "Master Kecamatan.csv")

df_out_kec.to_csv(OUT_KEC, index=False, encoding="utf-8-sig")
df_out_kab.to_csv(OUT_KAB, index=False, encoding="utf-8-sig")
df_out_master.to_csv(OUT_MASTER, index=False, encoding="utf-8-sig")

for p in (OUT_KEC, OUT_KAB, OUT_MASTER):
    assert os.path.isfile(p), f"Gagal membuat {p}"
    print("✓", p)

In [ ]:
# 17. CEK OUTPUT (baca ulang dari file)

cek = pd.read_csv(OUT_KEC, encoding="utf-8-sig", dtype={"kode_kecamatan": str})

kolom_diharapkan = [
    "kode_kecamatan", "kecamatan", "tahun", "tanaman", "luas_panen_ha", "produksi_ton",
    "produktivitas_kuintal_ha", "proporsi_alokasi", "bobot_sawah", "metode_alokasi",
    "is_estimate", "sumber",
]
assert list(cek.columns) == kolom_diharapkan
assert len(cek) == JUMLAH_KECAMATAN * jumlah_komoditas
assert cek["kecamatan"].nunique() == JUMLAH_KECAMATAN
assert cek["tanaman"].nunique() == jumlah_komoditas
assert cek["is_estimate"].all()
assert not cek.duplicated(["kecamatan", "tanaman"]).any()

print("✓ Struktur CSV sudah benar.")
print(f"✓ {len(cek)} baris, {cek['kecamatan'].nunique()} kecamatan, {cek['tanaman'].nunique()} komoditas.")
if TAHUN_DATA is None:
    print("⚠️  Ingat: kolom `tahun` masih kosong. Isi TAHUN_DATA lalu jalankan ulang.")

display(cek)